In [3]:
import os
from datetime import datetime
import regex as re
from collections import Counter

In [2]:
filename = "/kaggle/input/datasets/ffatty/plain-text-wikipedia-simpleenglish/2of2/wiki_00"

In [3]:
with open(filename) as f:
    text = f.read()

print(len(text))
print("Number of words: ", len(text.split(" ")))
print("--"*20)
print(text[0:1000])

1045423
Number of words:  169559
----------------------------------------
Academy of Urbanism

The Academy of Urbanism is a non-profit academic research organization dedicated to urban planning and urban development. The institute is located in London. It has branches in Europe and other places in the world. The Academy employs 500 professionals engaged in various subjects related to the field of urban planning and the chief of the Mossad motto is: "to recognize, encourage, and celebrate great places".

National workshops, lectures, an internet site and spend magazine. In addition, the Academy maintains Festival prizes and conducting congresses in the UK each year and operates a voluntary activity.


World Community Grid

World Community Grid (WCG) is a large public computing grid to work on scientific research projects that help people. People donate time on their personal computers to the project. The software can be set up to run only when the computer is not being used for other wo

In [1]:
class BasicTokenizer:

    def __init__(self, vocab_size):
        self.n_merges = vocab_size-256
        self.pattern = ''
        self.merges = {}
        self.special_tokens = ''
        self.vocab = {i: bytes([i]) for i in range(256)}
        
    @staticmethod
    def _get_stats(ids):
        return Counter(zip(ids, ids[1:]))
        
    @staticmethod
    def _merge(ids, pair, idx):
        i=0
        vocab = []
        while i < len(ids):
            if i<len(ids)-1 and ids[i]==pair[0] and ids[i+1]==pair[1]:
                vocab.append(idx)
                i=i+2
            else:
                vocab.append(ids[i])
                i=i+1
        return vocab

    def encode(self, text):
        ids = list(text.encode('utf-8'))
        for pair, idx in self.merges.items():
            ids = self._merge(ids, pair, idx) 
        return ids
        
    def decode(self, ids):
        tokens = b"".join([self.vocab[idx] for idx in ids])
        return tokens.decode('utf-8')
        
    def train(self,text, verbose=True):
        ids = list(text.encode("utf-8"))
        for i in range(self.n_merges):
            pair_counts = self._get_stats(ids)
            if not pair_counts:
                break
                
            best_pair = max(pair_counts, key=lambda x : pair_counts[x])
            idx = 256+i

            if verbose:
                print(f"Best pair at merge {i} is {best_pair} replaced with {idx}")
                
            self.merges[best_pair] = idx
            self.vocab[idx]=self.vocab[best_pair[0]] + self.vocab[best_pair[1]]
            ids = self._merge(ids, best_pair, idx)
        print(f"Training Completed!! at {i} merge")

    def train_regex_tokenizer(self, text, verbose):
        GPT4_SPLIT_PATTERN = r"""'(?i:[sdmt]|ll|ve|re)|[^\r\n\p{L}\p{N}]?+\p{L}+|\p{N}{1,3}| ?[^\s\p{L}\p{N}]++[\r\n]*|\s*[\r\n]|\s+(?!\S)|\s+"""
        compiled_pattern = re.compile(GPT4_SPLIT_PATTERN)

        txt_chunks = re.findall(compiled_pattern, text)

        ids = [list(txt.encode('utf-8')) for txt in txt_chunks]

        stats={}
        for i in range(self.n_merges):
            for chunk_ids in ids:
                self._get_stats(chunk_ids, stats)
                print(stats)
            pair_counts = stats
   
            if not pair_counts:
                break
                
            best_pair = max(pair_counts, key=lambda x : pair_counts[x])
            idx = 256+i

            if verbose:
                print(f"Best pair at merge {i} is {best_pair} replaced with {idx}")
                
            self.merges[best_pair] = idx
            self.vocab[idx]=self.vocab[best_pair[0]] + self.vocab[best_pair[1]]
            ids = self._merge(ids, best_pair, idx)
        print(f"Training Completed!! at {i} merge")
                
    def save(self):
            with open('vocab.txt', 'w') as f:
                for key, value in self.vocab.items():
                    f.write(f"{key}\t{value}\n")
    
            with open('merges.txt', 'w') as f:
                for key, value in self.merges.items():
                    f.write(f"{key}\t{value}\n")

In [5]:
start_time = datetime.now()
tokenizer = BasicTokenizer(vocab_size=4096)
tokenizer.train(text,verbose=False)
print(f"Took {datetime.now()-start_time}")

Training Completed!! at 3839 merge
Took 0:12:31.335868


# Tried to make GPT2 like Regex Tokeizer with GPT4 Regex patter

In [69]:
import regex as re
from collections import Counter

class RegexTokenizer:

    def __init__(self, vocab_size):
        self.n_merges = vocab_size-256
        self.merges ={}
        assert vocab_size > 256
        self.special_tokens = '<|endoftext|>'
        self.vocab = {i: bytes([i]) for i in range(256)}
        self.pattern = GPT4_SPLIT_PATTERN = r"""'(?i:[sdmt]|ll|ve|re)|[^\r\n\p{L}\p{N}]?+\p{L}+|\p{N}{1,3}| ?[^\s\p{L}\p{N}]++[\r\n]*|\s*[\r\n]|\s+(?!\S)|\s+"""
        self.compiled_pattern = re.compile(self.pattern)
        
    @staticmethod
    def _get_stats(ids):
        return Counter(zip(ids, ids[1:]))
        
    @staticmethod
    def _merge(ids, pair, idx):
        i=0
        vocab = []
        while i < len(ids):
            if i<len(ids)-1 and ids[i]==pair[0] and ids[i+1]==pair[1]:
                vocab.append(idx)
                i=i+2
            else:
                vocab.append(ids[i])
                i=i+1
        return vocab

    def encode(self, text):
        tokens=[]
        txt_chunks = re.findall(self.compiled_pattern, text)
        ids = [list(ch.encode('utf-8')) for ch in txt_chunks]
        
        for pair, idx in self.merges.items():
            ids =  [self._merge(ids_chunk, pair, idx) for ids_chunk in ids]

        [tokens.extend(i) for i in ids]
            
        return tokens
        
    def decode(self, ids):
        tokens = b"".join([self.vocab[idx] for idx in ids])
        return tokens.decode('utf-8')

    def train(self, text, verbose=False):
        
        txt_chunks = re.findall(self.compiled_pattern, text)
        ids = [list(txt.encode('utf-8')) for txt in txt_chunks]

        for i in range(self.n_merges):
            stats = Counter()
            for chunk_ids in ids:
                stats.update(self._get_stats(chunk_ids))
                if verbose:
                    print(stats)
            if not stats:
                break
                
            # best_pair = max(stats, key=stats.get)
            best_pair = stats.most_common(1)[0][0]
            idx = 256+i
            ids = [self._merge(chunks, best_pair, idx) for chunks in ids]

            if verbose:
                print(f"Best pair at merge {i} is {best_pair} replaced with {idx}")
                
            self.merges[best_pair] = idx
            self.vocab[idx]=self.vocab[best_pair[0]] + self.vocab[best_pair[1]]
            
        print(f"Training Completed!! at {i} merge")


    def save(self):
        with open('vocab.txt', 'w') as f:
            for key, value in self.vocab.items():
                f.write(f"{key}\t{value}\n")

        with open('merges.txt', 'w') as f:
            for key, value in self.merges.items():
                f.write(f"{key}\t{value}\n")

    def load(self,path):
        vocab_file = os.path.join(path, 'vocab.txt')
        merges_file = os.path.join(path, 'merges.txt')
        
        with open(vocab_file, 'r') as f:
            vocab = f.read()
            vocab = vocab.split("\n")[0:-1]
            vocab = {int(i.split('\t')[0]): eval(i.split("\t")[1]) for i in vocab}
            
        with open(merges_file, 'r') as f:
            merges = f.read()
            merges = merges.split("\n")[0:-1]
            merges = {eval(i.split("\t")[0]):eval(i.split("\t")[1]) for i in merges}

        self.merges = merges
        self.vocab = vocab

In [11]:
from datetime import datetime
start_time = datetime.now()
tokenize = RegexTokenizer(vocab_size=256+50000)
tokenize.train(text)
tokenize.save()
print(datetime.now()-start_time)

Training Completed!! at 1 merge
0:00:02.305346


In [70]:
start_time = datetime.now()
tokenize = RegexTokenizer(vocab_size=256+50000)
tokenize.load('/kaggle/input/notebooks/shubham219/gpt2-like-tokenizer')

In [75]:
print(tokenize.encode("hello world!!!:-)"))
print(tokenize.decode([257, 859, 111, 1085, 10925, 33, 58, 45, 41]))

[257, 859, 111, 1085, 10925, 33, 58, 45, 41]
hello world!!!:-)
